# Data cleaning: Corporate Financial Risk Assessment
**Lilly · CAP 3764 · Fall 2026**

This notebook documents how the raw Kaggle dataset becomes `data/processed/cleaned_data.csv`. It runs the team's shared cleaning code (`risk_data.clean`) instead of re-implementing it, shows the data before and after, and states the missing-value strategy. No models are built here.

Run all cells with **Python (cap3764-risk)** from the repository root or the `notebooks/` folder, after downloading the raw data with `python src/risk_data/collect.py`. Running the notebook rewrites `data/processed/cleaned_data.csv`.

## 1. Load the raw data
The cleaning module keeps its folder locations in `RAW_DATA_DIR` and `PROCESSED_DATA_DIR`, so the notebook reuses them. Kagglehub saves the CSV in a dataset subfolder, so we search the raw folder recursively and skip Jupyter checkpoint copies.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from risk_data.clean import PROCESSED_DATA_DIR, RAW_DATA_DIR, clean

raw_paths = sorted(p for p in Path(RAW_DATA_DIR).rglob("*.csv") if ".ipynb_checkpoints" not in p.parts)
assert raw_paths, "No raw CSV found. Run `python src/risk_data/collect.py` first (see README)."

raw = pd.read_csv(raw_paths[0])
print("Raw file:", raw_paths[0].name)
print("Raw rows and columns:", raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("Raw data type"))

Raw file: Corporate_Financial_Risk_Assessment_Data.csv
Raw rows and columns: (5000, 20)


,Company_ID,Date,Industry_Sector,Total_Assets,Total_Liabilities,Current_Assets,Current_Liabilities,Net_Income,Revenue,Operating_Income,Cash_Flow,Debt_Equity_Ratio,Return_on_Assets,Working_Capital_Ratio,Stock_Price_Close,Volatility_Index,GDP_Growth_Rate,Interest_Rate,Inflation_Rate,Financial_Risk_Label
0,C5706,2024-06-30,Finance,8.464139e+08,3.242897e+08,91702381.06,6176096.46,6.454550e+07,3.455171e+08,5.873791e+07,6.454030e+07,0.62,0.0763,14.85,478.57,20.57,7.17,7.66,3.09,0
1,C1577,2022-12-31,Technology,6.674340e+08,4.267892e+08,11734194.94,10982065.94,1.105900e+08,2.782570e+08,4.730370e+07,1.106493e+08,1.77,0.1657,1.07,577.31,35.17,6.77,6.46,1.54,0
2,C5016,2016-06-30,Energy,4.637573e+08,2.961599e+08,9983638.68,3811938.05,6.467575e+07,2.115831e+08,4.443246e+07,6.468172e+07,1.77,0.1395,2.62,463.87,33.94,2.07,9.42,6.67,0
3,C2817,2019-03-31,Manufacturing,3.592200e+08,4.980808e+07,93621021.75,49646893.27,3.085521e+07,3.130124e+07,2.504099e+06,3.077613e+07,0.16,0.0859,1.89,116.60,24.74,-4.97,1.09,4.29,0
4,C2287,2015-12-31,Manufacturing,9.776274e+08,6.220905e+08,72479345.81,35533291.66,6.517940e+07,1.593143e+07,4.460800e+06,6.524728e+07,1.75,0.0667,2.04,986.99,22.13,5.49,1.62,4.02,0


,Raw data type
Company_ID,str
Date,str
Industry_Sector,str
Total_Assets,float64
Total_Liabilities,float64
Current_Assets,float64
Current_Liabilities,float64
Net_Income,float64
Revenue,float64
Operating_Income,float64


## 2. Audit the raw data
Before changing anything we check for missing values and duplicates. The cleaning module treats two rows as duplicates when they share the same `Company_ID` and `Date`, and keeps the first one. If duplicate pairs disagreed on their values, keeping the first would silently discard different information, so we also count **conflicting** pairs.

In [2]:
key = ["Company_ID", "Date"]
numeric_raw = raw.select_dtypes(include="number").columns.tolist()
categorical_raw = ["Industry_Sector", "Financial_Risk_Label"]

missing_by_column = raw.isna().sum().to_frame("Missing values")
display(missing_by_column)

audit = pd.Series({
    "Raw rows": len(raw),
    "Raw columns": raw.shape[1],
    "Rows repeating an earlier Company_ID + Date": int(raw.duplicated(key).sum()),
    "Fully identical repeated rows": int(raw.duplicated().sum()),
    "Repeated keys with different values (conflicts)": int(raw.drop_duplicates().duplicated(key).sum()),
    "Missing numeric cells": int(raw[numeric_raw].isna().sum().sum()),
    "Missing categorical cells": int(raw[categorical_raw].isna().sum().sum()),
}, name="Value").to_frame()
display(audit)

,Missing values
Company_ID,0
Date,0
Industry_Sector,0
Total_Assets,0
Total_Liabilities,0
Current_Assets,0
Current_Liabilities,0
Net_Income,0
Revenue,0
Operating_Income,0


,Value
Raw rows,5000
Raw columns,20
Rows repeating an earlier Company_ID + Date,1425
Fully identical repeated rows,1425
Repeated keys with different values (conflicts),0
Missing numeric cells,0
Missing categorical cells,0


In [3]:
n_key_dups = int(audit.loc["Rows repeating an earlier Company_ID + Date", "Value"])
n_conflicts = int(audit.loc["Repeated keys with different values (conflicts)", "Value"])
n_missing_total = int(raw.isna().sum().sum())

conflict_note = (
    "None of them disagree, so keeping the first copy loses no information."
    if n_conflicts == 0
    else f"{n_conflicts} of them have different values from the row that was kept, so the team should decide whether keeping the first copy is acceptable."
)
display(Markdown(
    f"The raw file has **{len(raw):,} rows** and **{n_missing_total:,} missing cells**. "
    f"**{n_key_dups:,} rows** repeat an earlier `Company_ID` + `Date` pair. {conflict_note}"
))

The raw file has **5,000 rows** and **0 missing cells**. **1,425 rows** repeat an earlier `Company_ID` + `Date` pair. None of them disagree, so keeping the first copy loses no information.

## 3. Run the shared cleaning function
`clean()` loads the raw CSV, calls `clean_data()`, and saves the result to `data/processed/cleaned_data.csv`. Its steps are:

1. print the missing values per column,
2. drop duplicate `Company_ID` + `Date` rows (keeping the first),
3. convert `Date` to datetime and `Industry_Sector` and `Financial_Risk_Label` to the category type,
4. fill missing numeric values with the column median,
5. run two sanity checks (no duplicate keys, no missing numeric values), which stop the code with an error if cleaning failed.

The printed output below comes from the module itself.

In [4]:
cleaned = clean()
print("Cleaned rows and columns:", cleaned.shape)

Missing values per column:
Company_ID               0
Date                     0
Industry_Sector          0
Total_Assets             0
Total_Liabilities        0
Current_Assets           0
Current_Liabilities      0
Net_Income               0
Revenue                  0
Operating_Income         0
Cash_Flow                0
Debt_Equity_Ratio        0
Return_on_Assets         0
Working_Capital_Ratio    0
Stock_Price_Close        0
Volatility_Index         0
GDP_Growth_Rate          0
Interest_Rate            0
Inflation_Rate           0
Financial_Risk_Label     0
dtype: int64
Dropped 1425 duplicate rows
Cleaned rows and columns: (3575, 20)


## 4. Before and after

In [5]:
numeric_clean = cleaned.select_dtypes(include="number").columns.tolist()

summary = pd.DataFrame({
    "Raw": [len(raw), raw.shape[1], int(raw.duplicated(key).sum()), int(raw.isna().sum().sum())],
    "Cleaned": [len(cleaned), cleaned.shape[1], int(cleaned.duplicated(key).sum()), int(cleaned.isna().sum().sum())],
}, index=["Rows", "Columns", "Rows repeating a Company_ID + Date", "Missing cells"])
display(summary)

types = pd.DataFrame({"Raw type": raw.dtypes.astype(str), "Cleaned type": cleaned.dtypes.astype(str)})
changed = types[types["Raw type"] != types["Cleaned type"]].rename_axis("Column")
display(Markdown("**Columns whose data type changed:**"))
display(changed)

,Raw,Cleaned
Rows,5000,3575
Columns,20,20
Rows repeating a Company_ID + Date,1425,0
Missing cells,0,0


**Columns whose data type changed:**

,Raw type,Cleaned type
Column,,
Date,str,datetime64[us]
Industry_Sector,str,category
Financial_Risk_Label,int64,category


## 5. Missing-value strategy
This is the rule the team follows, and what the data showed.

| Column group | Rule | Implemented in `clean_data()`? |
|--------------|------|-------------------------------|
| **Numeric** (financial figures, ratios, macro indicators) | Fill missing values with the **column median**. The median is not pulled around by the large outliers that financial figures usually have. | Yes, step 4 |
| **Categorical** (`Industry_Sector`) | Do **not** guess. A missing sector would be reported and filled with an explicit `"Unknown"` category, so it stays visible. The mode is not used. | No, because none were found |
| **Target** (`Financial_Risk_Label`) | **Never impute the label.** A row with a missing label would be dropped and the count reported. | No, because none were found |
| **Identifiers** (`Company_ID`, `Date`) | Rows without them cannot be matched or de-duplicated, so they would be dropped and reported. | No, because none were found |

Two limits to keep in mind:
- The median is computed on the **whole** dataset. If a later notebook needs imputation for modeling, it should learn the median from the **training rows only** (for example inside a scikit-learn `Pipeline`), to avoid leakage.
- If a future version of the dataset has missing categorical, label or identifier values, `clean_data()` must be extended before it can be trusted. The sanity checks only cover numeric columns.

In [6]:
n_numeric_filled = int(raw[numeric_raw].isna().sum().sum())
n_categorical_missing = int(raw[categorical_raw].isna().sum().sum())
n_id_missing = int(raw[key].isna().sum().sum())

display(Markdown(
    f"In this dataset the median fill replaced **{n_numeric_filled:,}** numeric cells, and there were "
    f"**{n_categorical_missing:,}** missing categorical or label cells and **{n_id_missing:,}** missing identifier cells."
))
if n_categorical_missing or n_id_missing:
    display(Markdown("**Warning:** the categorical, label or identifier rules above are not implemented in `clean_data()`. Review before using the cleaned file."))

In this dataset the median fill replaced **0** numeric cells, and there were **0** missing categorical or label cells and **0** missing identifier cells.

## 6. Verify the result
These checks are independent of the ones inside `clean_data()`. They confirm that the saved file matches the cleaned data and that the changes are exactly the ones described above.

In [7]:
saved = pd.read_csv(Path(PROCESSED_DATA_DIR) / "cleaned_data.csv")

assert saved.shape == cleaned.shape, "Saved file shape differs from cleaned data"
assert not cleaned.duplicated(key).any(), "Duplicate Company_ID + Date pairs remain"
assert len(raw) - len(cleaned) == int(raw.duplicated(key).sum()), "Rows were removed for a reason other than duplicates"
assert cleaned.isna().sum().sum() == 0, "Missing values remain"
assert pd.api.types.is_datetime64_any_dtype(cleaned["Date"]), "Date is not a datetime column"
assert cleaned["Industry_Sector"].dtype == "category" and cleaned["Financial_Risk_Label"].dtype == "category"
assert set(cleaned["Financial_Risk_Label"].astype(int).unique()) == {0, 1}, "Expected binary risk labels"

risk_share = cleaned["Financial_Risk_Label"].astype(int).mean()
display(Markdown(
    f"All checks passed. The cleaned data has **{len(cleaned):,} rows x {cleaned.shape[1]} columns**, "
    f"dates from **{cleaned['Date'].min():%Y-%m-%d}** to **{cleaned['Date'].max():%Y-%m-%d}**, "
    f"and **{risk_share:.1%}** of companies labeled at risk."
))

All checks passed. The cleaned data has **3,575 rows x 20 columns**, dates from **2015-03-31** to **2024-12-31**, and **16.1%** of companies labeled at risk.

## 7. Summary and hand-off
The numbers quoted above come from this notebook's own output. The cleaned file is saved at `data/processed/cleaned_data.csv` and is what the EDA and modeling notebooks load. This notebook does not change the cleaning code. If the audit in section 2 reports conflicting duplicates, raise it with the team before relying on "keep the first row".